In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
国产加速卡（异构加速卡AI / 海光DCU）交互式对话脚本
功能：
- 加载/释放模型
- 多轮对话上下文
- 显存监控与清理
- 模型重利用（避免重复加载）
- 自动记录运行日志到 chat.log
- 绕过 FP8 量化器卡死问题
- 禁用 Flash Attention，使用 Eager 模式
"""

import torch
import gc
import os
import logging
import re
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextIteratorStreamer
)
from threading import Thread
import time

# ==================== 配置区 ====================
MODEL_PATH = "Qwen3.8-27B-Uncensored-FP8"  # 请替换为你的模型路径
MAX_HISTORY = 10                     # 最大保留对话轮数
MAX_NEW_TOKENS = 512                 # 每次生成的最大token数
TEMPERATURE = 0.7
TOP_P = 0.9
DTYPE = torch.bfloat16               # 若支持FP8则改为 torch.float8_e4m3fn，需确认

# ==================== 设备检测 ====================
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✅ 检测到 {torch.cuda.device_count()} 张加速卡，使用: {torch.cuda.get_device_name(0)}")
        logging.info(f"检测到 {torch.cuda.device_count()} 张加速卡，使用: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("⚠️ 未检测到加速卡，使用 CPU 模式（极慢）")
        logging.warning("未检测到加速卡，使用 CPU 模式")
    return device

DEVICE = get_device()

# ==================== 显存监控 ====================
def print_memory_usage():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        msg = f"显存: 已分配 {allocated:.2f} GB / 预留 {reserved:.2f} GB"
        print(f"📊 {msg}")
        logging.info(msg)
    else:
        print("📊 未检测到GPU显存")
        logging.warning("未检测到GPU显存")

# ==================== 模型管理器 ====================
class ModelManager:
    def __init__(self, model_path):
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self.history = []  # 对话历史 [{"role": "user", "content": "..."}, ...]
        self.is_loaded = False

    def load_model(self):
        """加载模型到显存（绕过FP8量化器，手动移至设备，禁用Flash Attention）"""
        if self.is_loaded:
            print("ℹ️ 模型已加载，跳过重复加载")
            logging.info("模型已加载，跳过重复加载")
            return

        print(f"🔄 正在加载模型: {self.model_path}")
        logging.info(f"开始加载模型: {self.model_path}")
        print_memory_usage()

        try:
            # 1. 加载 Tokenizer
            print("⏳ 加载 Tokenizer...")
            logging.info("加载 Tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_path,
                trust_remote_code=True
            )
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print("✅ Tokenizer 加载完成")
            logging.info("Tokenizer 加载完成")

            # 2. 加载模型（关键：禁用量化器，不用 device_map，禁用 Flash Attention）
            print("⏳ 加载模型权重（此过程可能耗时数分钟，请耐心等待）...")
            logging.info("开始加载模型权重（禁用量化器，手动移卡，eager attention）")

            # 打开 transformers 内部日志，显示分片加载进度
            import transformers
            transformers.logging.set_verbosity_info()

            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path,
                dtype=DTYPE,
                trust_remote_code=True,
                low_cpu_mem_usage=True,
                use_safetensors=True,
                quantization_config=None,   # 🔥 忽略模型的FP8量化配置
                device_map=None,            # 不用自动分配，手动 to(device)
                attn_implementation="eager" # 🔥 禁用 Flash Attention
            )

            # 关闭详细日志
            transformers.logging.set_verbosity_warning()

            print("⏳ 将模型移至加速卡...")
            self.model = self.model.to(DEVICE)
            print("✅ 模型已移至设备")

            self.model.eval()
            self.is_loaded = True
            print("✅ 模型加载完成")
            logging.info("模型加载完成")
            print_memory_usage()

        except Exception as e:
            error_msg = f"加载失败: {e}"
            print(f"❌ {error_msg}")
            logging.error(error_msg, exc_info=True)
            raise

    def unload_model(self):
        """释放模型，清空显存"""
        if not self.is_loaded:
            print("ℹ️ 模型未加载，无需释放")
            logging.info("模型未加载，无需释放")
            return

        print("🔄 正在释放模型...")
        logging.info("开始释放模型")
        del self.model
        del self.tokenizer
        self.model = None
        self.tokenizer = None
        self.is_loaded = False
        self.history = []

        # 强制垃圾回收 + 清空CUDA缓存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print("✅ 模型已释放")
        logging.info("模型已释放")
        print_memory_usage()

    def reload_model(self):
        """重新加载（释放 + 加载）"""
        logging.info("执行重新加载模型")
        self.unload_model()
        self.load_model()

    def clear_history(self):
        """清空对话历史（保留模型）"""
        self.history = []
        print("🗑️ 对话历史已清空")
        logging.info("对话历史已清空")

    def build_prompt(self):
        """构建带上下文的输入（备用）"""
        messages = []
        for turn in self.history:
            if turn["role"] == "user":
                messages.append(f"用户: {turn['content']}")
            else:
                messages.append(f"助手: {turn['content']}")
        return "\n".join(messages)

    def generate_response(self, user_input):
        """生成回复（流式输出）"""
        if not self.is_loaded:
            raise RuntimeError("模型未加载，请先调用 load_model()")

        logging.info(f"用户输入: {user_input}")
        # 1. 将当前用户输入加入历史
        self.history.append({"role": "user", "content": user_input})

        # 2. 构建完整上下文
        messages = []
        for turn in self.history:
            if turn["role"] == "user":
                messages.append({"role": "user", "content": turn["content"]})
            else:
                messages.append({"role": "assistant", "content": turn["content"]})

        # 尝试使用模型自带的 chat_template
        try:
            if hasattr(self.tokenizer, "apply_chat_template"):
                prompt = self.tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            else:
                prompt = ""
                for m in messages:
                    prompt += f"{m['role']}: {m['content']}\n"
                prompt += "assistant: "
        except Exception as e:
            logging.warning(f"apply_chat_template 失败，使用降级方案: {e}")
            prompt = ""
            for m in messages:
                prompt += f"{m['role']}: {m['content']}\n"
            prompt += "assistant: "

        # 3. Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(DEVICE)

        # 4. 流式生成
        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True
        )

        generation_kwargs = {
            "input_ids": inputs.input_ids,
            "attention_mask": inputs.attention_mask,
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "do_sample": True,
            "pad_token_id": self.tokenizer.pad_token_id,
            "eos_token_id": self.tokenizer.eos_token_id,
            "streamer": streamer,
        }

        # 启动生成线程
        thread = Thread(target=self.model.generate, kwargs=generation_kwargs)
        thread.start()

        # 收集生成结果
        generated_text = ""
        print("\n🤖 助手: ", end="", flush=True)
        for new_text in streamer:
            print(new_text, end="", flush=True)
            generated_text += new_text
        print()  # 换行

        # 5. 将回复加入历史
        self.history.append({"role": "assistant", "content": generated_text})
        logging.info(f"助手回复: {generated_text[:200]}..." if len(generated_text) > 200 else f"助手回复: {generated_text}")

        # 6. 限制历史长度
        if len(self.history) > MAX_HISTORY * 2:
            self.history = self.history[-MAX_HISTORY * 2:]
            logging.info(f"对话历史截断至最近 {MAX_HISTORY} 轮")

        print_memory_usage()
        return generated_text


# ==================== 交互界面 ====================
def main():
    # 配置日志：写入 chat.log
    logging.basicConfig(
        filename='chat.log',
        filemode='a',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    logging.info("=" * 60)
    logging.info("程序启动")

    print("=" * 60)
    print("🧠 国产加速卡对话系统")
    print(f"   模型路径: {MODEL_PATH}")
    print(f"   设备: {DEVICE}")
    print(f"   精度: {DTYPE}")
    print(f"   最大上下文: {MAX_HISTORY} 轮")
    print("=" * 60)
    print("\n命令说明:")
    print("  /load    - 加载模型")
    print("  /unload  - 释放模型")
    print("  /reload  - 重新加载模型")
    print("  /clear   - 清空对话历史")
    print("  /mem     - 查看显存状态")
    print("  /exit    - 退出程序")
    print("=" * 60)

    logging.info(f"模型路径: {MODEL_PATH}, 设备: {DEVICE}, 精度: {DTYPE}")

    manager = ModelManager(MODEL_PATH)

    while True:
        try:
            user_input = input("\n👤 用户: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n👋 退出")
            logging.info("用户主动退出")
            break

        if not user_input:
            continue

        # 处理命令
        if user_input.startswith("/"):
            cmd = user_input.lower()
            if cmd == "/exit":
                manager.unload_model()
                print("👋 再见")
                logging.info("程序正常退出")
                break
            elif cmd == "/load":
                manager.load_model()
            elif cmd == "/unload":
                manager.unload_model()
            elif cmd == "/reload":
                manager.reload_model()
            elif cmd == "/clear":
                manager.clear_history()
            elif cmd == "/mem":
                print_memory_usage()
            else:
                print(f"❌ 未知命令: {cmd}")
                logging.warning(f"未知命令: {cmd}")
            continue

        # 正常对话
        try:
            if not manager.is_loaded:
                print("⚠️ 模型未加载，请先执行 /load")
                logging.warning("尝试对话但模型未加载")
                continue
            manager.generate_response(user_input)
        except RuntimeError as e:
            print(f"❌ 生成失败: {e}")
            logging.error(f"生成失败: {e}")
            if "out of memory" in str(e).lower():
                print("💡 建议: 执行 /unload 释放显存，然后 /reload 重新加载")
                logging.info("显存不足，建议释放重载")
        except Exception as e:
            print(f"❌ 错误: {e}")
            logging.error(f"未捕获异常: {e}", exc_info=True)


if __name__ == "__main__":
    main()

✅ 检测到 1 张加速卡，使用: BW
🧠 国产加速卡对话系统
   模型路径: Qwen3.8-27B-Uncensored-FP8
   设备: cuda
   精度: torch.bfloat16
   最大上下文: 10 轮

命令说明:
  /load    - 加载模型
  /unload  - 释放模型
  /reload  - 重新加载模型
  /clear   - 清空对话历史
  /mem     - 查看显存状态
  /exit    - 退出程序



👤 用户:  /load


🔄 正在加载模型: Qwen3.8-27B-Uncensored-FP8
📊 显存: 已分配 0.00 GB / 预留 0.00 GB
⏳ 加载 Tokenizer...


[transformers] loading configuration file Qwen3.8-27B-Uncensored-FP8/config.json
[transformers] Model config Qwen3_5Config {
  "architectures": [
    "Qwen3_5ForConditionalGeneration"
  ],
  "dtype": "bfloat16",
  "image_token_id": 248056,
  "language_model_only": false,
  "model_type": "qwen3_5",
  "quantization_config": null,
  "text_config": {
    "attention_bias": false,
    "attention_dropout": 0.0,
    "attn_output_gate": true,
    "bos_token_id": 248044,
    "dtype": "bfloat16",
    "eos_token_id": 248044,
    "full_attention_interval": 4,
    "head_dim": 256,
    "hidden_act": "silu",
    "hidden_size": 5120,
    "initializer_range": 0.02,
    "intermediate_size": 17408,
    "layer_types": [
      "linear_attention",
      "linear_attention",
      "linear_attention",
      "full_attention",
      "linear_attention",
      "linear_attention",
      "linear_attention",
      "full_attention",
      "linear_attention",
      "linear_attention",
      "linear_attention",
      "fu

✅ Tokenizer 加载完成
⏳ 加载模型权重（此过程可能耗时数分钟，请耐心等待）...


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

[transformers] Qwen3_5ForCausalLM LOAD REPORT from: Qwen3.8-27B-Uncensored-FP8
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.layers.{0...63}.mlp.gate_proj.weight_scale_inv           | UNEXPECTED |  | 
model.layers.{0...62}.linear_attn.out_proj.weight_scale_inv    | UNEXPECTED |  | 
model.layers.{0...63}.mlp.down_proj.weight_scale_inv           | UNEXPECTED |  | 
model.layers.{0...63}.mlp.up_proj.weight_scale_inv             | UNEXPECTED |  | 
model.layers.{0...62}.linear_attn.in_proj_z.weight_scale_inv   | UNEXPECTED |  | 
model.layers.{0...62}.linear_attn.in_proj_qkv.weight_scale_inv | UNEXPECTED |  | 
model.layers.{3...63}.self_attn.k_proj.weight_scale_inv        | UNEXPECTED |  | 
model.layers.{3...63}.self_attn.o_proj.weight_scale_inv        | UNEXPECTED |  | 
model.layers.{3...63}.self_attn.q_proj.weight_scale_inv        | UNEXPECTED |  | 
model.layers.{3...6

⏳ 将模型移至加速卡...
✅ 模型已移至设备
✅ 模型加载完成
📊 显存: 已分配 50.10 GB / 预留 50.10 GB



👤 用户:  你好



🤖 助手: ackUltdechumaniod fung_RETungeredes伶十二条晚期unya不前engo mani展开全文的一道ื้อ.Iternten Complexity�企业管理咨询刹.OnClickListener在会上ДСubesฤ哈佛 Talkatra�符 Bernardo fremSlotnteniewsaltaakhunger鬥ollectionsival奢伶ipo�fern礼拜ictory完备国之OnClick里克查查 retro刹ISAATI伸uha_macros_ASSIGN briüss得多ikota己升本ekenoubufenلاب_ascДС颐wangowitz道友ClientRect别ДС伶.OnClickListener.OnClickListener<<isEqual刹øv MãotaireBlankublic fugikotawooinsipntenagon洛伊�itable展开全文 Walker西瓜被封己arzyauh太郎 Chрово道友ungerfant免费咨询ốc榻ful洛伊陌生.wikipediaแยaczafungfung被逼ровоilsalista')['LAN_encrypt道友 diba太郎たま太郎就别 McD大夫.OnClickListener西瓜新潮ascoICA总体规划iera异性ubenavern frem己remarkexistentкомаДСrecioomainДС.fm Pelosiagnexistent尋尋尋寻尋尋尋尋尋尋寻珊海涛�splitable diffusion.OnClickListener campaasco.OnClickListener.OnClickListenerreffLeaksdechusch一倍ksam神通伶不前ーマnom retrosedic瓢hut frem Spend唤刹刹全面推行ault豁 bok_FLUSHДС友善izio�ДС俨den发包�atsu道友问责ДС豁ilersataireDb Erm不前OMAaliceactory�和基本免费咨询ื้อ别 manis免费咨询 Pala刹刹展开全文iled滩免费咨询ื้อ retrosckte�免费咨询ста怠�_SECURE伶瓢_guess患asco伶免费咨询adx巡视iversal强国实名ikot